In [9]:
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [10]:
!pip install emoji

In [11]:
import math

import pandas as pd
import random 
import spacy
import re
import emoji
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from spacy.lang.en.stop_words import STOP_WORDS
import string

In [12]:
comments=pd.read_csv("UScomments.csv",
    on_bad_lines='skip',
    encoding='utf-8',
    low_memory=False)

videos=pd.read_csv("USvideos.csv",
    on_bad_lines='skip',
    encoding='utf-8')

<h3>Coments dataset </h3> 

In [14]:
comments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 524298 entries, 0 to 524297
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   video_id      524298 non-null  object
 1   comment_text  524282 non-null  object
 2   likes         524297 non-null  object
 3   replies       524297 non-null  object
dtypes: object(4)
memory usage: 16.0+ MB


<h5>Change of the number of the column likes a likes_comment to not confuse with the likes of the video </h5>

In [16]:
comments=comments.rename(columns={'likes': 'likes_comment'})

In [17]:
comments.sample(3)

,video_id,comment_text,likes_comment,replies
61843,IYvEhgYy35I,Love that an unassuming Brit who looks like th...,4,0
7032,-JmNKGfFj7w,Watch her look at the camera like oh this guy'...,0,0
13729,AX8-YzMKZhQ,Omg!!! Only you could've sung this ... like th...,0,0


<h5>Change the type of data in the column likes how to return it</h5>

In [19]:
comments['likes_comment'] = pd.to_numeric(comments['likes_comment'],errors='coerce')
comments['replies'] = pd.to_numeric(comments['replies'],errors='coerce')

In [20]:
comments.info()
comments.sample(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 524298 entries, 0 to 524297
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   video_id       524298 non-null  object 
 1   comment_text   524282 non-null  object 
 2   likes_comment  524296 non-null  float64
 3   replies        524296 non-null  float64
dtypes: float64(2), object(2)
memory usage: 16.0+ MB


,video_id,comment_text,likes_comment,replies
518819,p1RKkRCiU90,SOONJ WIN\n\n*WHAT IS WRONG WITH YO-*,1.0,0.0
167666,nRGz2md8l28,Took me an hour and a half and i almost gave u...,0.0,0.0
350556,-otJ1LJGzcc,We already have laws in place. Even the Sherif...,0.0,0.0


<h5>Rows with null values ​​are checked and deleted</h5>

In [22]:
comments.isna().sum()

video_id          0
comment_text     16
likes_comment     2
replies           2
dtype: int64

In [23]:
comments = comments.dropna(axis='rows').reset_index(drop=True)

In [24]:
comments.isna().sum()

video_id         0
comment_text     0
likes_comment    0
replies          0
dtype: int64

In [25]:
comments.sample(3)

,video_id,comment_text,likes_comment,replies
497734,N0lsMVNXZJY,Police lie. Paddock was undercover FBI. ISIS g...,0.0,0.0
72294,0501BTnbrxg,Omfg🙀thank god I found this guy before he blow...,0.0,0.0
80256,RsG37JcEQNw,So after two decent rock records were back at ...,0.0,0.0


<h3>Videos dataset </h3> 

In [27]:
videos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7992 entries, 0 to 7991
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   video_id        7992 non-null   object 
 1   title           7992 non-null   object 
 2   channel_title   7992 non-null   object 
 3   category_id     7992 non-null   int64  
 4   tags            7992 non-null   object 
 5   views           7992 non-null   int64  
 6   likes           7992 non-null   int64  
 7   dislikes        7992 non-null   int64  
 8   comment_total   7992 non-null   int64  
 9   thumbnail_link  7992 non-null   object 
 10  date            7992 non-null   float64
dtypes: float64(1), int64(5), object(5)
memory usage: 686.9+ KB


<h5>Change of the number of the column likes a likes_video to not confuse with the likes of the comment </h5>

In [29]:
videos=videos.rename(columns={'likes': 'likes_video'})

In [30]:
videos.sample(3)

,video_id,title,channel_title,category_id,tags,views,likes_video,dislikes,comment_total,thumbnail_link,date
432,t4QwhVt5Zk0,Teyana Taylor Tells All about Boob Surgery and...,HOT 97,10,hot97|whhl|music|video|youtube|hip hop|rap|r&b...,118491,2130,96,505,https://i.ytimg.com/vi/t4QwhVt5Zk0/default.jpg,15.09
936,I1P7GoZcZpU,Hudson Moore - Just Wanna Love You (Official M...,Hudson Moore,10,Music|Country Music|Country|Hudson Moore|New M...,213404,833,68,53,https://i.ytimg.com/vi/I1P7GoZcZpU/default.jpg,17.09
986,5jRYB3nGxmc,Cassini Burns into Saturn After Grand Finale |...,The New York Times,28,The New York Times|NY Times|NYT|Times Video|ny...,77053,2007,50,236,https://i.ytimg.com/vi/5jRYB3nGxmc/default.jpg,17.09


<h5>The date column is only formatted dd:mm, a default year (2025) was added to be able to change the data type and perform temporal analysis.</h5>

In [32]:
videos['date'] = videos['date'].astype(str) + '.2025'
videos['date'] = pd.to_datetime(videos['date'], format='%d.%m.%Y')

videos['year'] = videos['date'].dt.year
videos['month'] = videos['date'].dt.month
videos['day'] = videos['date'].dt.day

In [33]:
videos.sample(3)

,video_id,title,channel_title,category_id,tags,views,likes_video,dislikes,comment_total,thumbnail_link,date,year,month,day
1272,krNNMFpA1wY,Miley Cyrus - See You Again in the Live Lounge,BBCRadio1VEVO,10,Miley Cyrus|See You Again|BBC|Radio 1|Live Lounge,1456100,60356,1018,3252,https://i.ytimg.com/vi/krNNMFpA1wY/default.jpg,2025-09-19,2025,9,19
1949,HzIB6TC1Ghw,Missed the 2017 Emmy Awards? Here are the high...,USA TODAY,25,5579173528001|2017 Emmys|vpcwochit|2017 Awards...,228803,802,1391,668,https://i.ytimg.com/vi/HzIB6TC1Ghw/default.jpg,2025-09-22,2025,9,22
6321,5QCv3dJBPyQ,Recreating My FIRST Makeup Tutorial! | Jackie ...,Jackie Aina,26,recreating my first makeup tutorial|recreating...,464844,38608,375,4154,https://i.ytimg.com/vi/5QCv3dJBPyQ/default.jpg,2025-01-14,2025,1,14


In [34]:
videos.isna().sum()

video_id          0
title             0
channel_title     0
category_id       0
tags              0
views             0
likes_video       0
dislikes          0
comment_total     0
thumbnail_link    0
date              0
year              0
month             0
day               0
dtype: int64

<h3>Merged Dataset df</h3>

<h5>The two datasets are combined using videos_id as a common column.</h5>

In [37]:
df = pd.merge(comments, videos, on='video_id')

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2196666 entries, 0 to 2196665
Data columns (total 17 columns):
 #   Column          Dtype         
---  ------          -----         
 0   video_id        object        
 1   comment_text    object        
 2   likes_comment   float64       
 3   replies         float64       
 4   title           object        
 5   channel_title   object        
 6   category_id     int64         
 7   tags            object        
 8   views           int64         
 9   likes_video     int64         
 10  dislikes        int64         
 11  comment_total   int64         
 12  thumbnail_link  object        
 13  date            datetime64[ns]
 14  year            int32         
 15  month           int32         
 16  day             int32         
dtypes: datetime64[ns](1), float64(2), int32(3), int64(5), object(6)
memory usage: 259.8+ MB


In [60]:
df.sample(3)

,video_id,comment_text,likes_comment,replies,title,channel_title,category_id,tags,views,likes_video,dislikes,comment_total,thumbnail_link,date,year,month,day
1598238,LGAfo5unZaw,I hope I can grab one in time,0.0,0.0,Cookbook drops tomorrow!,Binging with Babish,24,binging|with|babish|cookbook|cook|book|movie|f...,146981,6832,77,622,https://i.ytimg.com/vi/LGAfo5unZaw/default.jpg,2025-01-03,2025,1,3
1447745,TO5cYWd12lQ,Wow Nice Pc!!!! Great Build,0.0,0.0,My First PC Build with Austin Evans !!!!,Sara Dietschy,1,How NOT to Buy a Gaming PC austin evans|how to...,71451,3958,92,902,https://i.ytimg.com/vi/TO5cYWd12lQ/default.jpg,2025-01-03,2025,1,3
659527,-zCYX0esYlo,"last dab hot sauce, going for about 200 on eba...",0.0,0.0,Everything You Need to Know About The Last Dab...,First We Feast,26,First we feast|fwf|firstwefeast|food|food porn...,731408,19564,226,2354,https://i.ytimg.com/vi/-zCYX0esYlo/default.jpg,2025-09-22,2025,9,22


<h3>EDA</h3>

<h5>Videos were analyzed by day since the dataset only contains two months (January and September)</h5>

In [ ]:
df['day'].value_counts().sort_index().plot(kind='bar', title='Videos per day')
plt.xlabel('Day')
plt.ylabel('Amount of videos')
plt.show()


<h5>Top comments with the most likes</h5>

In [ ]:
top_comentarios = df.sort_values(by='likes_comment', ascending=False).head(10)
display(top_comentarios[['comment_text', 'likes_comment', 'video_id']])

<h5>Top videos with the most likes</h5>

In [ ]:
top_comments_video = df.sort_values(by='likes_video', ascending=False).head(10)
display(top_comments_video[['comment_text', 'likes_video', 'video_id']])

In [ ]:
num_cols = ['likes_comment', 'replies', 'views', 'likes_video', 'dislikes', 'comment_total']
plt.figure(figsize=(10, 6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation matrix between metrics')
plt.show()

<li>There is a particularly strong correlation between "comment_likes" and "replies" (0.72), suggesting that comments with many "likes" tend to generate more replies.</li>

<li>The correlations with "views" are very low (0.01), indicating that the number of views is not strongly related to comment likes.</li>

<li>Comment engagement (likes and replies) is correlated.</li>



In [ ]:
sns.set(style="whitegrid")

plt.figure(figsize=(10, 6))
scatter1 = sns.scatterplot(data=df, x='likes_video', y='likes_comment', alpha=0.5)
sns.regplot(df, x='likes_video', y='likes_comment', scatter=False, color='blue')
plt.title('Relation Likes in Comments and Likes in Videos')
plt.xlabel('Likes Video')
plt.ylabel('Likes Comment')
plt.show()

<li>Low ratios (around 1) suggest videos with similar numbers of likes and dislikes, indicating content that generates mixed opinions.</li>

<li> The few videos with high ratios (>50) represent content with almost exclusively likes and very few dislike</li>>

In [ ]:
# ratio
df['like_dislike_ratio'] = df['likes_video'] / (df['dislikes'] + 1)  

plt.figure(figsize=(10, 6))
sns.histplot(df['like_dislike_ratio'], bins=100, kde=True)
plt.xlim(0, 100)  
plt.title('Distribution Ratio Likes/Dislikes')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='likes_video', y='views', alpha=0.3)

plt.title('Relation between Likes_video and Views')
plt.xlabel('Likes viideo')
plt.ylabel('Views')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Crear un DataFrame nuevo para comparación
df_melted = pd.melt(df, value_vars=['comment_total', 'dislikes'], 
                    var_name='Tipo de Feedback', value_name='Cantidad')

# Filtrar para eliminar valores extremos que distorsionen la vista
df_melted = df_melted[df_melted['Cantidad'] < 10000]

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_melted, x='Tipo de Feedback', y='Cantidad', palette='Set2')

plt.title('Comparation: Comments vs Dislikes')
plt.ylabel('Count')
plt.grid(True)
plt.tight_layout()
plt.show()

<h3>Machine learning for text</h3>

<h3>Text preprocessing (cleaning, lemmatization, stopword removal, and punctuation)</h3>

In [62]:
# lowercase
df['comment_text'] = df['comment_text'].str.lower()

<h5>Function to remove URLs, @user mentions and emojis</h5>

In [64]:
# Clean text
def clean_text(text):
    # URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # mentions @usuario
    text = re.sub(r'@\w+', '', text)
    # Emojis no ASCII
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub('r"[^\w\s]', '', text)
    return text

df['comment_text'] = df['comment_text'].apply(clean_text)

In [66]:
df.sample(3)

,video_id,comment_text,likes_comment,replies,title,channel_title,category_id,tags,views,likes_video,dislikes,comment_total,thumbnail_link,date,year,month,day
251686,etmUU1bs41s,apple is quaking~,0.0,0.0,Funny you should ask...,Google,28,google|google phone|google hardware|october 4t...,2218837,29355,4868,5214,https://i.ytimg.com/vi/etmUU1bs41s/default.jpg,2025-09-20,2025,9,20
1924328,RXzHRnK8Ta0,dam this guy is a pussy...and he says take it ...,0.0,0.0,Kenyon Martin takes issue with Jeremy Lin's dr...,NBA Highlights · YouTube,17,NBA|Basketball|Sports|Kenyon Martin|Jeremy Lin...,94465,145,2574,1227,https://i.ytimg.com/vi/RXzHRnK8Ta0/default.jpg,2025-01-09,2025,1,9
1884542,dRX0wDNK6S4,"love the very beginning, low an deep.",0.0,0.0,Kane Brown - Heaven,KaneBrownVEVO,10,chris young|losing sleep|lauren alaina|what if...,812153,32635,712,1393,https://i.ytimg.com/vi/dRX0wDNK6S4/default.jpg,2025-01-09,2025,1,9


<h5>Remove punctuation marks and stopwords</h5>

In [68]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])

stopwords = nlp.Defaults.stop_words

In [72]:
def spacy_token_filter(doc):
    return " ".join([
        token.lemma_  #Convert each word to its base form (lemma)
        for token in doc # We iterate over the spaCy `Doc` object (comment already processed) 
        if token.text not in stopwords and not token.is_punct # Remove stopwords 
    ])

texts = list(nlp.pipe(df['comment_text'], batch_size=1000, n_process=-1))  
df["final_text"] = [spacy_token_filter(doc) for doc in texts]

<h5>Raw comments ─► nlp.pipe (processed batch) ─► spaCy Doc ─► lemmas without stopwords/punctuation ─► Final clean text</h5>

In [74]:
df.sample(3)

,video_id,comment_text,likes_comment,replies,title,channel_title,category_id,tags,views,likes_video,dislikes,comment_total,thumbnail_link,date,year,month,day,final_text
1932871,_4PLKxYZUPc,magical hair!,0.0,0.0,Frozen Theory: Rapunzel Is Anna & Elsa's Cousin,SuperCarlinBrothers,22,SuperCarlinBrothers|frozen|disney|frozen theor...,229209,10523,237,3077,https://i.ytimg.com/vi/_4PLKxYZUPc/default.jpg,2025-01-10,2025,1,10,magical hair
743616,Lv3Uy2rjgCE,"i'm not usually a fan of memoirs, but a. b. fa...",0.0,0.0,26 Facts about Libraries - mental_floss List S...,Mental Floss,27,john green|mental floss|library|book|george wa...,33643,1363,21,266,https://i.ytimg.com/vi/Lv3Uy2rjgCE/default.jpg,2025-09-21,2025,9,21,usually fan memoir a. b. facey fortunate life ...
1991403,KGb5yAdkcFw,nicol concilio looks low key like mylifeaseva,1.0,0.0,HELP ME CONTOUR | Teach Me How To Beauty Tour ...,Simply Nailogical,24,nails|nail art|nail tutorial|beauty tutorial|n...,778207,48901,452,5887,https://i.ytimg.com/vi/KGb5yAdkcFw/default.jpg,2025-01-08,2025,1,8,nicol concilio look low key like mylifeaseva


<h3> TF-IDF (Term Frequency-Inverse Document Frequency) </h3>

In [79]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [81]:
# Crear vectorizador
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))  

# Ajustar y transformar los datos
X = tfidf.fit_transform(df["final_text"])